# 🧠 Activation Functions and Their Derivatives

Welcome to the hands-on explanation notebook for **Activation Functions**! In this notebook, we will:
1. Define the mathematical formulas for Sigmoid, ReLU, Leaky ReLU, and SiLU (Swish).
2. Implement these activations and their derivatives from scratch in NumPy.
3. Plot and compare the activations side-by-side.
4. Plot and compare their derivatives to visualize the **Vanishing Gradient** and **Dying ReLU** problems.
5. Explain why YOLO defaults to using **SiLU** in its convolutional layers.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Custom Activations and Derivatives in NumPy

Let's implement the functions and their derivatives.
- **Sigmoid:** $\sigma(x) = \frac{1}{1 + e^{-x}}$
- **ReLU:** $\max(0, x)$
- **Leaky ReLU:** $\max(0.1x, x)$
- **SiLU:** $x \cdot \sigma(x)$

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1.0 - s)

def relu(x):
    return np.maximum(0.0, x)

def relu_derivative(x):
    return np.where(x > 0.0, 1.0, 0.0)

def leaky_relu(x, alpha=0.1):
    return np.maximum(alpha * x, x)

def leaky_relu_derivative(x, alpha=0.1):
    return np.where(x > 0.0, 1.0, alpha)

def silu(x):
    return x * sigmoid(x)

def silu_derivative(x):
    sig = sigmoid(x)
    return sig * (1.0 + x * (1.0 - sig))

## 2. Visualizing Activations vs. Derivatives

Let's plot the activation functions on the left, and their derivatives on the right, to analyze backpropagation gradient flow.

In [ ]:
x = np.linspace(-6, 6, 300)

plt.figure(figsize=(16, 7))

# Plot Activation Functions
plt.subplot(1, 2, 1)
plt.plot(x, sigmoid(x), color='purple', linewidth=2.5, label='Sigmoid')
plt.plot(x, relu(x), color='red', linewidth=2.5, label='ReLU')
plt.plot(x, leaky_relu(x), color='green', linewidth=2.5, label='Leaky ReLU (α=0.1)')
plt.plot(x, silu(x), color='teal', linewidth=3, label='SiLU (Swish)')
plt.ylim(-2, 5)
plt.xlabel('Input (x)')
plt.ylabel('Activation Output')
plt.title('Activation Functions')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

# Plot Derivatives
plt.subplot(1, 2, 2)
plt.plot(x, sigmoid_derivative(x), color='purple', linewidth=2.5, label='Sigmoid Derivative')
plt.plot(x, relu_derivative(x), color='red', linewidth=2.5, label='ReLU Derivative')
plt.plot(x, leaky_relu_derivative(x), color='green', linewidth=2.5, label='Leaky ReLU Derivative')
plt.plot(x, silu_derivative(x), color='teal', linewidth=3, label='SiLU Derivative')
plt.ylim(-0.2, 1.2)
plt.xlabel('Input (x)')
plt.ylabel('Gradient / Slope')
plt.title('Derivatives (Gradient Flow)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

## 3. Key Observations & Diagnoses

*   **Sigmoid (Vanishing Gradient):** For $|x| > 4$, the derivative drops to near zero. In deep networks, multiplying these tiny values layer-by-layer causes the initial layers' gradients to vanish entirely, stopping learning.
*   **ReLU (Dying ReLU):** For all negative inputs $x < 0$, the derivative is exactly 0. If a neuron gets updated such that it always receives negative values, it will never update again ("dies").
*   **Leaky ReLU (Dying ReLU Prevention):** Keeps a small slope ($0.1$ or $0.01$) for negative values, allowing gradients to flow back even for inactive neurons.
*   **SiLU (Smooth Gradient):** The default in YOLO. It has a smooth curve with a continuous derivative (no sharp corner at $x=0$) and a small negative valley. This prevents dead units while stabilizing batch updates.